# Notebook 04: Structured Outputs & JSON Schema

**Objectives:**
- Extract structured data with json_extract.v1 prompt
- Validate against JSON schemas
- Repair malformed JSON
- Log token costs for validation + repair cycles

In [1]:
import sys
import pprint
sys.path.append('..')

from utils.prompts import render
from utils.llm_client import LLMClient
from utils.logging_utils import log_llm_call
from utils.router import pick_model
from utils.json_utils import safe_parse_json, validate_json_schema, create_simple_schema, format_schema_for_prompt
import json

import re
import pandas as pd
from pathlib import Path

## Part 1: JSON Extraction with Schema

Define a schema and extract structured data.

In [13]:
data_path = Path('../data/News Feed.txt')
news_feed = [
    block.strip()
    for block in data_path.read_text(encoding='utf-8').split('\n')
    if block.strip()
]

schema = create_simple_schema({
                            "district": "string",
                            "flood_level_meters": "float",
                            "victim_count": "int",
                            "main_need": "string",
                            "status": "string"
                        }, required=[])

model = pick_model('openai', 'general')
client = LLMClient('openai', model)

for news in news_feed:
    prompt_text, spec = render("json_extract.v1", schema=schema, text=news)

    response = client.json_chat(
        [
            {"role": "user", "content": prompt_text}
        ],
        temperature=0.0
    )['text']
    success, data, error = safe_parse_json(response)
    pprint.pprint(data)

{'district': 'Colombo',
 'flood_level_meters': 9.5,
 'main_need': 'assistance',
 'status': 'critical',
 'victim_count': 0}
{'district': 'Ja-Ela',
 'flood_level_meters': None,
 'main_need': 'boat',
 'status': 'urgent',
 'victim_count': 5}
{'district': 'Peradeniya',
 'flood_level_meters': 0.0,
 'main_need': 'None',
 'status': 'Traffic moving slowly',
 'victim_count': 0}
{'district': 'Kalutara',
 'flood_level_meters': 0.0,
 'main_need': 'Rescue team',
 'status': 'urgent',
 'victim_count': 12}
{'district': 'Gampaha',
 'flood_level_meters': 2.0,
 'main_need': 'dry rations',
 'status': 'fully underwater',
 'victim_count': 500}
{}
{'district': 'Matara',
 'flood_level_meters': 0.0,
 'main_need': 'none',
 'status': 'stable',
 'victim_count': 0}
{'district': 'Beddagana',
 'flood_level_meters': 1.5,
 'main_need': 'rescue',
 'status': 'urgent',
 'victim_count': 1}
{'district': 'Galle fort area',
 'flood_level_meters': 0.0,
 'main_need': 'none',
 'status': 'safe',
 'victim_count': 0}
{'district': '

KeyboardInterrupt: 

Handle malformed JSON with automatic repair.

## Part 2: Pydantic Models (Recommended Approach)

Use Pydantic for type-safe structured outputs with automatic validation and IDE support.


In [4]:
from pydantic import BaseModel, Field
from utils.json_utils import (
                            format_pydantic_schema_for_prompt,
                            parse_json_with_pydantic
)

class CrisisEvent(BaseModel):
    district: str = Field(..., description="The name of the district")
    flood_level_meters: float = Field(..., description="The current flood level in meters")
    victim_count: bool = Field(..., description="The number of victims affected by the flood")
    main_need: bool = Field(..., description="The main need of the affected people (e.g., food, shelter, medical aid)")
    status: bool = Field(..., description="The current status of the flood situation (e.g., critical, stable, warning)")


schema_str = format_pydantic_schema_for_prompt(CrisisEvent)
print(schema_str)

{
  "properties": {
    "district": {
      "description": "The name of the district",
      "title": "District",
      "type": "string"
    },
    "flood_level_meters": {
      "description": "The current flood level in meters",
      "title": "Flood Level Meters",
      "type": "number"
    },
    "victim_count": {
      "description": "The number of victims affected by the flood",
      "title": "Victim Count",
      "type": "boolean"
    },
    "main_need": {
      "description": "The main need of the affected people (e.g., food, shelter, medical aid)",
      "title": "Main Need",
      "type": "boolean"
    },
    "status": {
      "description": "The current status of the flood situation (e.g., critical, stable, warning)",
      "title": "Status",
      "type": "boolean"
    }
  },
  "required": [
    "district",
    "flood_level_meters",
    "victim_count",
    "main_need",
    "status"
  ],
  "title": "CrisisEvent",
  "type": "object"
}


In [ ]:
from pandas.io.formats.printing import _pprint_seq

# Extracted objects will convert to dataframe and save as excel
extracted_data = []

for news in news_feed:
    prompt_text, spec = render("json_extract.v1", schema=schema_str, text=news)

    response = client.json_chat(
        [
            {"role": "user", "content": prompt_text}
        ],
        temperature=0.0
    )['text']
    if success:
        s, event, err = parse_json_with_pydantic(response, CrisisEvent)
        if s:
            extracted_data.append(event)
        else:
            print(f"Failed to parse with pydantic: {err}")
    else:
        print(f"Failed to parse JSON: {error}")
    
# Convert to DataFrame and save as  in ../output/flood_repot.xlsx
df = pd.DataFrame([e.dict() for e in extracted_data])
output_path = Path('../output/flood_report.xlsx')



In [16]:
df.to_excel(output_path, index=False)
print(f"Extracted data saved to {output_path}")

Extracted data saved to ..\output\flood_report.xlsx
